# Model Lab · Agent
**Illinois State University · IT344**

Compare models for an applied project: quality, cost, response time, speed, and alternatives similar to a model you already use.

1. Choose **File → Save a copy in Drive** to keep your own notebook.
2. Choose **Runtime → Change runtime type → T4 GPU**, if available. The local Qwen agent downloads about 3 GB; CPU runs are slower.
3. Open **Secrets** (the key icon in Colab’s left sidebar). Add a secret named **AA_API_KEY**, paste your Artificial Analysis key as its value, and turn **Notebook access** on for this notebook. Obtain a key through [Artificial Analysis](https://artificialanalysis.ai/documentation).
4. Choose **Runtime → Run all**. The data-loading cell reads `AA_API_KEY` using `google.colab.userdata.get`.
5. Use the embedded app or its temporary Gradio link. Keep this runtime running while you work.

The key stays in your Colab Secrets and is read only to fetch a fresh snapshot. It is not printed or copied into notebook source, app state, or downloads. Each student uses their own secret and enables access for their own notebook copy. Do not paste the key into a code cell. The temporary app link can be opened by anyone who has it; it serves the snapshot and analysis, not your key. The agent runs Qwen inside your runtime; no LLM API key is needed. It is a checked tool workflow, not a general chatbot.


## 1. Set up
Run this cell once per runtime. It normally takes a few minutes.


In [ ]:
# Install dependencies and load the small shared classroom source package.
import io
import os
from pathlib import Path
import subprocess
import ssl
import sys
import tempfile
import urllib.request
import zipfile

os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
packages = ['gradio==6.0.2', 'pandas>=2.2,<3', 'plotly>=5.24,<7', 'certifi>=2024.2.2', 'huggingface-hub>=0.34,<1', 'transformers==4.57.6', 'torch>=2.6,<3', 'safetensors>=0.4,<1']
print("Installing app dependencies…")
installed = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], capture_output=True, text=True)
if installed.returncode:
    print(installed.stderr[-3000:])
    raise RuntimeError("Installation failed. Reconnect to a fresh Colab runtime and rerun this cell.")

# Only these Python files are loaded; the AA data comes from your own key below.
source_dir = Path(tempfile.mkdtemp(prefix="it344-colab-"))
source_url = "https://codeload.github.com/elaheJ/it344-model-lab/zip/refs/heads/main"
needed = ["__init__.py", "core.py", "snapshots.py", "agent.py", "local_qwen.py", "colab_ui.py"]
import certifi
context = ssl.create_default_context()
context.load_verify_locations(cafile=certifi.where())
with urllib.request.urlopen(source_url, timeout=60, context=context) as response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
with archive:
    for filename in needed:
        member = "it344-model-lab-main/model_compare/" + filename
        target = source_dir / "model_compare" / filename
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))
sys.path.insert(0, str(source_dir))
print("Ready. Continue to load AA data from Colab Secrets.")


## 2. Load fresh AA data
This cell reads **AA_API_KEY** from **Colab Secrets**. Enable **Notebook access** for the secret before running it. Missing, empty, or inaccessible secrets produce setup instructions without asking you to type a key. If the AA request fails, check the secret value, API access/quota and connection, then rerun this cell. No older snapshot is substituted. To refresh later, rerun this cell and the launch cell.


In [ ]:
from model_compare.colab_ui import load_colab_snapshot

snapshot = None  # Clear an earlier snapshot before attempting a fresh download.
snapshot = load_colab_snapshot()


## 3. Load the local agent
This downloads the public Qwen2.5 1.5B instruction model, then reuses it for later runs. GPU availability is controlled by Colab. If you only want to inspect the scripted workflow, skip this cell and choose **Scripted rehearsal (no LLM)** in the app.


In [ ]:
from model_compare.local_qwen import load_local_controller

if snapshot is not None:
    controller = load_local_controller()
    print("Agent ready on", controller["device"])
else:
    print("Load AA data successfully before downloading the agent model.")


## 4. Open the app
Use **Compare** for single or weighted metrics, and **Find similar models** for one or more features. All comparable alternatives are included. Expand **Project workload** to estimate costs for your own token volumes.


In [ ]:
from model_compare.colab_ui import launch_app

if "app" in globals():
    app.close()  # Rerunning this cell replaces the earlier app and snapshot.
if snapshot is None:
    print("Return to the AA data cell and load data successfully first.")
else:
    app = launch_app(snapshot, agent=True)


## Try it for your project
- Set your workload and limits, then compare models. Explain the tradeoff behind your shortlist.
- Change the ranking to **Weighted composite**. Adjust weights and explain any change in the winner.
- Pick a reference model and find alternatives similar in cost and TTFT. Add a quality feature and compare the results.
- Download the CSV and JSON evidence with your snapshot date. Evaluate the shortlisted models using actual project inputs.

**Notes:** rankings require quality, TTFT and cost measurements, plus any selected metric. Similarity only requires the selected features. Missing measurements are not zero. Range-normalized scores are relative to this snapshot. Benchmark end-to-end times reflect AA's workload, not a prediction of your project's latency.

If the embedded app is blank, open the printed Gradio link. If a campus filter blocks that link too, ask IT to review it. Colab sessions and share links are temporary; download results before disconnecting. This notebook is for interactive coursework, not permanent hosting. See [Gradio sharing](https://www.gradio.app/guides/sharing-your-app) and [Colab runtime limits](https://research.google.com/colaboratory/faq.html).

[Classroom source](https://github.com/elaheJ/it344-model-lab) · [Developer source and tests](https://github.com/elaheJ/ai-model-comparison-using-artificial-analysis-dev)
